# Prior visualisation — PD (Probability of Default)

What our synthetic prior produces for **PD**, next to the unmodified TabICL
prior, and next to the real datasets.

The quantity that matters here is **base rate — how rare default is**.

Real credit portfolios default at 7–40%, never 50%. TabICL's prior cuts its
latent at a random row, so its minority rate is roughly Uniform(0,1) — imbalance
there is **uncontrolled, not missing**, which is a weaker claim than the project's
original framing and worth stating honestly.

Measured across all 14 PD datasets: base rates run from **6.7%** to **40%**.
Every one sits below the balance point the original prior centres on.

### How the prior does it

Our prior assigns defaults with the **Merton/Vasicek one-factor model** — the
basis of the Basel IRB formula:

```
A_i = sqrt(rho) * Z + sqrt(1 - rho) * eps_i
default_i = 1 if A_i < Phi^-1(PD)
```

`rho` is sampled over Basel's prescribed **0.03–0.24** asset correlations (QRRE
0.04, mortgage 0.15, corporate 0.12–0.24). Two things this gives that a plain
quantile cut does not: the base rate is *exact*, and defaults are **correlated**
through `Z`, so the realised rate varies between cohorts. A prior of independent
labels has never shown the model a bad year.

`eps_i` is the SCM latent, **not** fresh noise — that is what keeps the features
predictive. Using noise there would give a perfect base rate and an unlearnable task.

### How to read this notebook

Right now there are **two** priors per task: `original` (unmodified TabICL, the control)
and `credit_v1` (ours). Everything below discovers whatever variants exist, so adding a
`credit_v2` needs no edit here.

By default it generates **500 datasets per prior locally** to show the structure. That is
plenty for every plot — it is not the pool the model trains on, and the notebook says so.

The companion notebooks are `prior_visualisation_lgd.ipynb` (the other task) and
`data_exploration.ipynb` (the real datasets these are aimed at).

All logic lives in `src/visualize/`; this notebook holds none. **It ends with a text
summary you can copy straight into a message.**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.visualize import pool_plots as pp, prior_plots, style, summaries

style.use_style()
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK  = "pd"
N     = 500     # datasets to draw per prior — enough to show the structure
FOCUS = None    # variant for the detail plots; None = the first non-original one
SAVE  = False   # True -> also write PNGs to results/_local/figures/

### What the colours mean

`original` is always grey (the control) and our prior is always blue, in **every** figure
in this notebook and its sibling — so you never have to re-read a legend.

In [ ]:
style.show_palette();

## 1. Which priors are on this machine?

`COMPLETE` means the whole pool the model trained on. `SAMPLE` means a partial copy —
fine for every plot here, but never quote it as "the pool".

If nothing is found, the next cell **generates the two arms live** and labels them
`(live)`. That is the normal path on a laptop.

In [ ]:
variants = pp.discover_pools(TASK)
print(f"pools found: {variants or 'none — will generate live'}")
pp.describe_pools(TASK) if variants else None

## 2. Draw the datasets

Same seed for every variant, so the draws are comparable rather than independently lucky.

In [ ]:
loaded, SOURCE = pp.load_variants_or_generate(TASK, n=N, seed=0)
print(f"source = {SOURCE}")
print({k: len(v) for k, v in loaded.items()})

if FOCUS is None or FOCUS not in loaded:
    non_original = [v for v in loaded if not v.startswith("original")]
    FOCUS = non_original[0] if non_original else list(loaded)[0]
print(f"FOCUS (detail plots) = {FOCUS}")

REFERENCE = pp.real_reference(TASK)

## 3. The summary table

One row per prior. For PD the columns that matter are the ones about
base rate.

Expect `original` and `credit_v1` to differ sharply. If they look the same, something is
wrong with the generation, not with the plot.

In [ ]:
pp.variant_summary(loaded, TASK)

## 4. The key comparison

The question to ask is **does our prior's cloud actually cover where the real datasets
sit** — not merely "is it different from the original".

Reference values are measured from your processed datasets when they are present, and
fall back to recorded values otherwise (the output says which).

In [ ]:
fig = pp.plot_target_comparison(loaded, TASK, reference=REFERENCE)

## 5. Target shapes, one row per prior

Ten histograms per variant. Look for **variety** within a row: a prior that always
produces the same shape will fit one real dataset and fail on the others, and that
failure is invisible in a mean.

In [ ]:
fig = pp.plot_target_shapes_by_variant(loaded, n_per=10)

## 6. Feature dependence

O'Prior's central claim is that what a prior teaches is a **dependence structure**, not
individual functions. Bold lines are medians.

If two variants' spectra sit on top of each other they teach a similar structure
*however different their targets look* — which would be an important negative result,
not a boring one.

In [ ]:
fig = pp.plot_spectrum_by_variant(loaded)

## 7. Shape sanity check

All variants should look **the same** here. Table shape is not what we are changing, so a
visible difference means an accidental confound rather than a finding.

In [ ]:
fig = pp.plot_shapes_by_variant(loaded)

## 8. One prior up close

Everything below uses `FOCUS` only — these are the plots that cannot be stacked across
variants. Change `FOCUS` in the setup cell and re-run this section to inspect another.

In [ ]:
focus_tasks = loaded[FOCUS]
print(f"{FOCUS}: {len(focus_tasks)} datasets")
fig = prior_plots.plot_target_grid(focus_tasks, n_show=min(100, len(focus_tasks)))

In [ ]:
fig = prior_plots.plot_table_shapes(focus_tasks)

In [ ]:
fig = prior_plots.plot_feature_relationships(focus_tasks, n_show=6)

In [ ]:
fig = prior_plots.plot_feature_target_relation(focus_tasks, n_show=8)

## 9. Save the figures (optional)

`results/_local/` is gitignored — regenerate rather than commit.

In [ ]:
if SAVE:
    print(style.savefig(pp.plot_target_comparison(loaded, TASK, reference=REFERENCE),
                        f"results/_local/figures/{TASK}_target_comparison.png"))
    print(style.savefig(pp.plot_target_shapes_by_variant(loaded),
                        f"results/_local/figures/{TASK}_shapes_by_variant.png"))
    print(style.savefig(pp.plot_spectrum_by_variant(loaded),
                        f"results/_local/figures/{TASK}_spectrum.png"))
else:
    print("SAVE is False — nothing written.")

---

## 10. TEXT SUMMARY

Everything above, as text. **Copy-paste this** — it carries the numbers, which the
figures do not.

In [ ]:
print(summaries.prior_summary(loaded, TASK, source=SOURCE, reference=REFERENCE))